# 0: Setting up the notebook

## Imports

In [1]:
from __future__ import annotations

import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

## Constants to help around with the files

In [2]:
RAW_FILES = {
    "players_stats": "epl_players_stats_2021_2026.csv",
    "players_shooting": "epl_players_shooting_2021_2026.csv",
    "keepers_stats": "epl_keepers_stats_2021_2026.csv",
    "squad_stats": "epl_squad_stats_2021_2026.csv",
    "squad_shooting": "epl_squad_shooting_2021_2026.csv",
    "squad_goalkeeping": "epl_squad_goalkeeping_2021_2026.csv",
    "squad_misc": "epl_squad_misc_2021_2026.csv",
    "squad_playingtime": "epl_squad_playingtime_2021_2026.csv",
    "opponent_stats": "epl_opponent_stats_2021_2026.csv",
    "opponent_shooting": "epl_oppenent_shooting_2021_2026.csv",
    "opponent_goalkeeping": "epl_oppenent_goalkeeping_2021_2026.csv",
    "opponent_misc": "epl_oppenent_misc_2021_2026.csv",
    "opponent_playingtime": "epl_oppenent_playingtime_2021_2026.csv",
    "rank_overall": "epl_rank_overall_2021_2026.csv",
    "rank_home_away": "epl_rank_home_away_2021_2026.csv",
}

MATCHES_OUT = "epl_matches_2021_2026.csv"
MATCHES_ODDS_OUT = "epl_matches_odds_extra_2021_2026.csv"

PEOPLE = ["players_stats", "players_shooting", "keepers_stats"]
TEAM_TABLES = [k for k in RAW_FILES if k not in PEOPLE]
OPPONENT_TABLES = [k for k in RAW_FILES if k.startswith("opponent_")]
EXPECTED_TEAMS_PER_SEASON = 20
ODDS_MIN_FILL = 0.95

MATCHES_RENAME = {
    "HomeTeam": "home_team", "AwayTeam": "away_team",
    "FTHG": "ft_home_goals", "FTAG": "ft_away_goals", "FTR": "ft_result",
    "HTHG": "ht_home_goals", "HTAG": "ht_away_goals", "HTR": "ht_result",
    "HS": "home_shots", "AS": "away_shots", "HST": "home_shots_on_target", "AST": "away_shots_on_target",
    "HF": "home_fouls", "AF": "away_fouls", "HC": "home_corners", "AC": "away_corners",
    "HY": "home_yellows", "AY": "away_yellows", "HR": "home_reds", "AR": "away_reds",
}
MATCH_CORE = ["Season", "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HTHG", "HTAG",
              "HTR", "Referee", "HS", "AS", "HST", "AST", "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR"]

## **Helper Functionalitly**

In [3]:
class Ctx:
    """Holds the data frames and the audit trail while the pipeline runs."""

    def __init__(self) -> None:
        self.frames: dict[str, pd.DataFrame] = {}
        self.orig_cols: dict[str, list[str]] = {}
        self.dropped: dict[str, list[tuple[str, str]]] = {}
        self.log: list[dict] = []
        self.canonical_teams: set[str] = set()

    def note(self, step: int, dataset: str, action: str, n: int | None = None, detail: str = "") -> None:
        self.log.append({"step": step, "dataset": dataset, "action": action, "rows_affected": n, "detail": detail})

    def drop(self, name: str, cols: list[str], reason: str) -> None:
        df = self.frames[name]
        present = [c for c in cols if c in df.columns]
        self.frames[name] = df.drop(columns=present)
        for c in present:
            self.dropped.setdefault(name, []).append((c, reason))

    def summary(self) -> pd.DataFrame:
        """Returns a high-level overview table of all datasets currently loaded."""
        data = []
        for name, df in self.frames.items():
            data.append({
                "dataset": name,
                "rows": df.shape[0],
                "columns": df.shape[1],
                "memory_kb": round(df.memory_usage(deep=True).sum() / 1024, 2)
            })
        if not data:
            return pd.DataFrame(columns=["dataset", "rows", "columns", "memory_kb"])
        return pd.DataFrame(data)

    def show_log(self, step: int | None = None, dataset: str | None = None) -> pd.DataFrame:
        """Returns the pipeline audit trail, optionally filtered by step or dataset."""
        if not self.log:
            return pd.DataFrame(columns=["step", "dataset", "action", "rows_affected", "detail"])
        
        filtered = self.log
        if step is not None:
            filtered = [entry for entry in filtered if entry["step"] == step]
        if dataset is not None:
            filtered = [entry for entry in filtered if entry["dataset"] == dataset]
            
        return pd.DataFrame(filtered)

    def show_dropped(self, dataset: str | None = None) -> pd.DataFrame:
        """Flattens dropped columns into an audit table, optionally filtered by dataset."""
        rows = []
        for ds, cols in self.dropped.items():
            if dataset is not None and ds != dataset:
                continue
            for col, reason in cols:
                rows.append({"dataset": ds, "column": col, "reason": reason})
        if not rows:
            return pd.DataFrame(columns=["dataset", "column", "reason"])
        return pd.DataFrame(rows)

    def show_teams(self) -> pd.DataFrame:
        """Returns the set of canonical teams as a sorted DataFrame."""
        if not self.canonical_teams:
            return pd.DataFrame(columns=["canonical_team"])
        return pd.DataFrame({"canonical_team": sorted(list(self.canonical_teams))})

### Helper Functions

In [4]:
def clean_header(c: str) -> str:
    """Remove stray BOM bytes (also in their mojibake form 'ï»¿') and whitespace from a header."""
    return c.replace("﻿", "").replace("ï»¿", "").strip()

In [5]:
def snake(col: str) -> str:
    """Raw header -> SQL/BI-friendly snake_case.   'Per 90 Minutes_G+A-PK' -> 'per_90_minutes_g_plus_a_minus_pk'."""
    special = {"# Pl": "n_players", "90s": "playing_time_90s"}  # '90s' = same measure as 'Playing Time_90s'
    if col in special:
        return special[col]
    s = col.strip()
    s = s.replace("+/-", " plus_minus ").replace("+", " plus ").replace("-", " minus ")
    s = s.replace("%", " pct ").replace("/", " per ").replace(">", " over ").replace("<", " under ").replace(".", "_")
    s = re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()
    return ("x_" + s) if s[0].isdigit() else s

In [6]:
def season_start(season: pd.Series) -> pd.Series:
    return season.str[:4].astype(int)

# 1. Loading Files

> read everything as text, repair the header, trim whitespace, blank -> NULL

## Implementation

In [7]:
def read_text_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

In [8]:
def coalesce_duplicate_headers(df: pd.DataFrame, ctx: Ctx, name: str) -> pd.DataFrame:
    """Matches.csv has the division code in two columns ('Div' and a BOM-polluted 'ï»¿Div'), each filled for
    different seasons. After cleaning the header both are called 'Div' -> merge them into one column."""
    cleaned = [clean_header(c) for c in df.columns]
    out: dict[str, pd.Series] = {}
    for raw, new in zip(df.columns, cleaned):
        s = df[raw]
        if new not in out:
            out[new] = s
            continue
        both = (out[new] != "") & (s != "")
        if (out[new][both] != s[both]).any():
            raise ValueError(f"{name}: duplicate columns {new!r} disagree")
        merged = out[new].where(out[new] != "", s)
        ctx.note(1, name, f"merged duplicate column '{new}'", int((s != "").sum()),
                 f"second copy supplied {int(((out[new] == '') & (s != '')).sum())} values")
        out[new] = merged
    return pd.DataFrame(out)

In [9]:
def normalise_cells(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Trim whitespace in every cell and turn empty strings into NULL."""
    changed = 0
    for c in df.columns:
        s = df[c]
        t = s.str.strip()
        changed += int((s != t).sum())
        df[c] = t.where(t != "", np.nan)
    return df, changed

In [10]:
raw_dir = Path("../data/raw/")
matches_path = Path("../data/raw/Matches.csv")
ctx = Ctx()

In [11]:
for name, fn in RAW_FILES.items():
    df = read_text_csv(raw_dir / fn)
    ctx.orig_cols[name] = list(df.columns)
    df, changed = normalise_cells(df)
    ctx.frames[name] = df
    ctx.note(1, name, "loaded", len(df), f"{df.shape[1]} columns; {changed} cells trimmed")
m = read_text_csv(matches_path)
ctx.orig_cols["matches"] = list(m.columns)
m = coalesce_duplicate_headers(m, ctx, "matches")
m, changed = normalise_cells(m)
ctx.frames["matches"] = m
ctx.note(1, "matches", "loaded", len(m), f"{m.shape[1]} columns; {changed} cells trimmed (e.g. 'J Gillett ')")

## Showing Result

In [12]:
ctx.summary()

,dataset,rows,columns,memory_kb
0,players_stats,2930,26,3927.14
1,players_shooting,2930,20,3020.30
2,keepers_stats,210,28,300.69
3,squad_stats,100,22,112.71
4,squad_shooting,100,14,72.52
5,squad_goalkeeping,100,22,111.74
6,squad_misc,100,16,81.07
7,squad_playingtime,100,20,102.69
8,opponent_stats,100,22,113.01
9,opponent_shooting,100,14,72.81


# 2. Structure

> drop repeated header rows inside the player/keeper files, drop junk columns

In [13]:
for name in PEOPLE:
    df = ctx.frames[name]
    # FBref repeats the column-header row every ~26 rows; those rows have Player == 'Player'.
    hdr = df["Player"] == "Player"
    ctx.note(2, name, "dropped repeated header rows", int(hdr.sum()))
    df = df[~hdr].reset_index(drop=True)
    ctx.frames[name] = df
    # Rk is a per-season row counter (alphabetical), 'Matches' is the text of a hyperlink.
    ctx.drop(name, ["Rk"], "row counter, not data")
    ctx.drop(name, ["Matches"], "hyperlink text, constant 'Matches'")

In [14]:
ctx.show_log(step=2)

,step,dataset,action,rows_affected,detail
0,2,players_stats,dropped repeated header rows,110,
1,2,players_shooting,dropped repeated header rows,110,
2,2,keepers_stats,dropped repeated header rows,5,


# 3. Team Names

> strip the "vs " prefix in opponent files; one canonical club-name list

In [15]:
for name in OPPONENT_TABLES:
    df = ctx.frames[name]
    if not df["Squad"].str.startswith("vs ").all():
        raise ValueError(f"{name}: expected every Squad to start with 'vs '")
    df["Squad"] = df["Squad"].str.removeprefix("vs ")
    ctx.note(3, name, "removed 'vs ' prefix from Squad", len(df),
             "row now describes what the opponents did AGAINST this squad")
ctx.canonical_teams = set(ctx.frames["squad_stats"]["Squad"])

In [16]:
ctx.show_teams()

,canonical_team
0,Arsenal
1,Aston Villa
2,Bournemouth
3,Brentford
4,Brighton
5,Burnley
6,Chelsea
7,Crystal Palace
8,Everton
9,Fulham


In [17]:
ctx.show_log(step=3)

,step,dataset,action,rows_affected,detail
0,3,opponent_stats,removed 'vs ' prefix from Squad,100,row now describes what the opponents did AGAIN...
1,3,opponent_shooting,removed 'vs ' prefix from Squad,100,row now describes what the opponents did AGAIN...
2,3,opponent_goalkeeping,removed 'vs ' prefix from Squad,100,row now describes what the opponents did AGAIN...
3,3,opponent_misc,removed 'vs ' prefix from Squad,100,row now describes what the opponents did AGAIN...
4,3,opponent_playingtime,removed 'vs ' prefix from Squad,100,row now describes what the opponents did AGAIN...


In [18]:
ctx.frames['opponent_stats']

,Squad,# Pl,Age,Poss,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,...,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Season
0,Arsenal,27,26.9,46.8,38,418,3420,38.0,47,33,...,5,6,69,3,1.24,0.87,2.11,1.11,1.97,2021-2022
1,Aston Villa,31,26.7,53.5,38,418,3420,38.0,52,35,...,4,6,83,1,1.37,0.92,2.29,1.26,2.18,2021-2022
2,Brentford,29,26.8,55.5,38,418,3420,38.0,54,41,...,5,5,50,3,1.42,1.08,2.5,1.29,2.37,2021-2022
3,Brighton,26,26.9,45.3,38,418,3420,38.0,44,34,...,3,4,80,0,1.16,0.89,2.05,1.08,1.97,2021-2022
4,Burnley,23,26.7,60.2,38,418,3420,38.0,51,36,...,5,5,44,0,1.34,0.95,2.29,1.21,2.16,2021-2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Nottingham,31,26.3,53.2,38,418,3420,38.0,48,34,...,5,6,68,1,1.26,0.89,2.16,1.13,2.03,2025-2026
96,Sunderland,29,26.3,55.1,38,418,3420,38.0,46,36,...,2,3,74,2,1.21,0.95,2.16,1.16,2.11,2025-2026
97,Tottenham,31,26.3,49.6,38,418,3420,38.0,57,42,...,3,3,70,1,1.5,1.11,2.61,1.42,2.53,2025-2026
98,West Ham,32,26.2,57.4,38,418,3420,38.0,62,42,...,6,7,75,0,1.63,1.11,2.74,1.47,2.58,2025-2026


In [19]:
for name in TEAM_TABLES:
    df = ctx.frames[name]
    if set(df["Squad"]) != ctx.canonical_teams:
        raise ValueError(f"{name}: club names differ from canonical set: {set(df['Squad']) ^ ctx.canonical_teams}")
    per_season = df.groupby("Season")["Squad"].nunique()
    if not (per_season == EXPECTED_TEAMS_PER_SEASON).all():
        raise ValueError(f"{name}: not 20 clubs in every season: {per_season.to_dict()}")
    if df.duplicated(["Squad", "Season"]).any():
        raise ValueError(f"{name}: duplicate (Squad, Season)")

In [20]:
for name in PEOPLE:
    extra = set(ctx.frames[name]["Squad"]) - ctx.canonical_teams
    if extra:
        raise ValueError(f"{name}: unknown clubs {extra}")
    ctx.frames[name]["Squad"] = ctx.frames[name]["Squad"].str.replace(r"\s+", " ", regex=True)
    ctx.frames[name]["Player"] = ctx.frames[name]["Player"].str.replace(r"\s+", " ", regex=True)

In [21]:
ctx.note(3, "all team tables", "club names verified", None,
             f"{len(ctx.canonical_teams)} canonical clubs, 20 per season, (Squad, Season) unique")

In [22]:
ctx.show_log(step=3)

,step,dataset,action,rows_affected,detail
0,3,opponent_stats,removed 'vs ' prefix from Squad,100.0,row now describes what the opponents did AGAIN...
1,3,opponent_shooting,removed 'vs ' prefix from Squad,100.0,row now describes what the opponents did AGAIN...
2,3,opponent_goalkeeping,removed 'vs ' prefix from Squad,100.0,row now describes what the opponents did AGAIN...
3,3,opponent_misc,removed 'vs ' prefix from Squad,100.0,row now describes what the opponents did AGAIN...
4,3,opponent_playingtime,removed 'vs ' prefix from Squad,100.0,row now describes what the opponents did AGAIN...
5,3,all team tables,club names verified,NaN,"27 canonical clubs, 20 per season, (Squad, Sea..."


# 4. Types

In [23]:
def cast_numeric(df: pd.DataFrame, skip: set[str] = frozenset()) -> int:
    """A column becomes numeric only if EVERY non-null value is a number.  It becomes an integer column
    only if every value is written as an integer in the source ('38' yes, '38.0' no) -> '90s' stays decimal."""
    n = 0
    for c in df.columns:
        if c in skip:
            continue
        s = df[c]
        nn = s.dropna()
        if nn.empty or pd.to_numeric(nn, errors="coerce").isna().any():
            continue
        if nn.str.fullmatch(r"-?\d+").all():
            df[c] = pd.to_numeric(s).astype("Int64")
        else:
            df[c] = pd.to_numeric(s).astype("float64")
        n += 1
    return n

In [24]:
for name in PEOPLE + TEAM_TABLES:
    n = cast_numeric(ctx.frames[name])
    ctx.note(4, name, "text -> numeric", n, "columns converted")

In [25]:
for df in ctx.frames.values():
    df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2820 entries, 0 to 2819
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Player                 2820 non-null   str    
 1   Nation                 2817 non-null   str    
 2   Pos                    2820 non-null   str    
 3   Squad                  2820 non-null   str    
 4   Age                    2819 non-null   Int64  
 5   Born                   2819 non-null   Int64  
 6   Playing Time_MP        2820 non-null   Int64  
 7   Playing Time_Starts    2820 non-null   Int64  
 8   Playing Time_Min       2820 non-null   Int64  
 9   Playing Time_90s       2820 non-null   float64
 10  Performance_Gls        2820 non-null   Int64  
 11  Performance_Ast        2820 non-null   Int64  
 12  Performance_G+A        2820 non-null   Int64  
 13  Performance_G-PK       2820 non-null   Int64  
 14  Performance_PK         2820 non-null   Int64  
 15  Performance_PKa

# 5. players & keepers

> players + keepers: split Nation, split Pos, add player_key and flags

In [26]:
ctx.frames['players_stats']['Nation'].unique()

<StringArray>
['eng ENG', 'sct SCO',  'dz ALG',  'no NOR',  'nl NED',  'es ESP',  'br BRA',
  'py PAR',  'co COL',  'gh GHA',  'dk DEN',  'jm JAM',  'fr FRA',  'ga GAB',
  'at AUT',  'ci CIV',  'us USA',  'gd GRN',  'pl POL',  'ba BIH',  'uy URU',
  'be BEL',  'is ISL',  'ml MLI',  'al ALB',  'ar ARG',  'ec ECU', 'wls WAL',
  'pt POR', 'nir NIR',  'bd BAN',  'ie IRL',  'ao ANG',  'cz CZE',  'zm ZAM',
  'ng NGA',  'ma MAR',  'sk SVK',  'se SWE',  'eg EGY',  'do DOM',  'fi FIN',
  'ir IRN',  'gr GRE',  'de GER',  'kr KOR',  'mx MEX',  'it ITA',  'tr TUR',
  'cd COD',  'gn GUI',  'sn SEN',  'hr CRO',  'cw CUW',  'rs SRB',  'cm CMR',
  'tn TUN',  'jp JPN',  'ua UKR',  'zw ZIM',       nan,  'xk KVX',  've VEN',
  'ch SUI',  'cl CHI',  'bf BFA',  'nz NZL',  'au AUS',  'cr CRC',  'il ISR',
  'tg TOG',  'ht HAI',  'gw GNB',  'ro ROU',  'za RSA',  'ca CAN',  'hu HUN',
  'iq IRQ',  'uz UZB',  'gm GAM',  'ms MSR',  'si SVN',  'tt TRI',  'lv LVA',
  'cy CYP',  'bg BUL',  'ge GEO',  'mz MOZ',  'bb 

In [27]:
for name in PEOPLE:
    df = ctx.frames[name]
    # Nation: 'eng ENG' = lowercase flag code + 3-letter code (1:1 mapping, checked) -> keep the 3-letter code.
    nat = df["Nation"].str.extract(r"^[a-z\-]+\s+([A-Z]{3})$")[0]
    if (nat.isna() != df["Nation"].isna()).any():
            raise ValueError(f"{name}: unexpected Nation format")
    ctx.note(5, name, "Nation -> 3-letter code", int(nat.notna().sum()),
             f"{int(nat.isna().sum())} blank Nation stay NULL (cannot be recovered from other seasons)")
    df["Nation"] = nat

    ctx.frames[name] = df

ctx.frames['players_stats']['Nation'].unique()

<StringArray>
['ENG', 'SCO', 'ALG', 'NOR', 'NED', 'ESP', 'BRA', 'PAR', 'COL', 'GHA', 'DEN',
 'JAM', 'FRA', 'GAB', 'AUT', 'CIV', 'USA', 'GRN', 'POL', 'BIH', 'URU', 'BEL',
 'ISL', 'MLI', 'ALB', 'ARG', 'ECU', 'WAL', 'POR', 'NIR', 'BAN', 'IRL', 'ANG',
 'CZE', 'ZAM', 'NGA', 'MAR', 'SVK', 'SWE', 'EGY', 'DOM', 'FIN', 'IRN', 'GRE',
 'GER', 'KOR', 'MEX', 'ITA', 'TUR', 'COD', 'GUI', 'SEN', 'CRO', 'CUW', 'SRB',
 'CMR', 'TUN', 'JPN', 'UKR', 'ZIM',   nan, 'KVX', 'VEN', 'SUI', 'CHI', 'BFA',
 'NZL', 'AUS', 'CRC', 'ISR', 'TOG', 'HAI', 'GNB', 'ROU', 'RSA', 'CAN', 'HUN',
 'IRQ', 'UZB', 'GAM', 'MSR', 'SVN', 'TRI', 'LVA', 'CYP', 'BUL', 'GEO', 'MOZ',
 'BRB', 'SUR', 'PER']
Length: 91, dtype: str

In [31]:
for name in PEOPLE:
    df = ctx.frames[name]
    parts = df["Pos"].str.split(",", expand=True)
    df["pos_primary"] = parts[0]
    df["pos_secondary"] = parts[1] if parts.shape[1] > 1 else np.nan
    ctx.note(5, name, "Pos split into pos_primary / pos_secondary", int(df["pos_secondary"].notna().sum()),
             "rows with two positions; original Pos kept")

    ctx.frames[name] = df

ctx.frames['players_stats'][['pos_primary', 'pos_secondary']]

,pos_primary,pos_secondary
0,DF,NaN
1,FW,NaN
2,MF,NaN
3,DF,NaN
4,DF,NaN
...,...,...
2815,DF,NaN
2816,DF,NaN
2817,FW,MF
2818,MF,NaN


In [ ]:
for name in PEOPLE:
    df = ctx.frames[name]
    # player_key: names alone are NOT unique ('Joshua King' = two different people, born 1992 and 2007).
    df["player_key"] = df["Player"] + "|" + df["Born"].astype("string").fillna("NA")
    if df.duplicated(["player_key", "Squad", "Season"]).any():
        raise ValueError(f"{name}: duplicate (player_key, Squad, Season)")
    nat_per_key = df.dropna(subset=["Nation"]).groupby("player_key")["Nation"].nunique()
    if (nat_per_key > 1).any():
        raise ValueError(f"{name}: one player_key with several nationalities")

In [32]:
a, b = ctx.frames["players_stats"], ctx.frames["players_shooting"]
k = ["Player", "Squad", "Season", "Born"]
aligned = a[k].astype("string").fillna("NA").equals(b[k].astype("string").fillna("NA"))
ctx.note(5, "players_shooting", "row alignment with players_stats", len(b),
         "identical keys in identical order" if aligned else "NOT aligned - merge on keys, not position")
if not aligned:
    raise ValueError("players_shooting is not row-aligned with players_stats")